In [2]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.cluster import KMeans
from PIL import Image
import os
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import shutil
from collections import Counter
import json

In [6]:
# load resNet 50 model
model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*(list(model.children())[:-1]))
model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.0406], std=[0.229, 0.224, 0.225])
])

c:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\User/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [01:17<00:00, 1.33MB/s]


In [9]:
def extract_features(path):
    img = Image.open(path).convert('RGB')
    t = transform(img).unsqueeze(0)
    with torch.no_grad():
        return model(t).flatten().numpy()


# paths
samples = r'C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\data\k-means_training\fabrics'
fabric_csv = "fabric_analysis_results.csv"
avg_features = {}

for fabric_type in os.listdir(samples):
    folder_path = os.path.join(samples, fabric_type)
    
    if os.path.isdir(folder_path):
        all_features = [extract_features(os.path.join(folder_path, f)) for f in os.listdir(folder_path)]
        avg_features[fabric_type] = np.mean(all_features, axis=0)

In [1]:
# analyzing the dataset
runway = r'C:\Users\User\Desktop\Lagos-FW-2025-Analysis\Lagos-FW-2025-Analysis\data\runway'
results = []

print("classifying runway fabrics using samples...")
for root, dirs, files in os.walk(runway):
    for img_name in files:
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_paths = os.path.join(root, img_name)

            try:
                curr_features = extract_features(img_paths)

                #best match to samples
                best_match = None
                min_dist = float('inf')

                for name, ref_features in avg_features.items():
                    dist = np.linalg.norm(curr_features - ref_features)
                    if dist < min_dist:
                        min_dist = dist
                        best_match = name
                
                brand_name = os.path.basename(root)
                results.append({
                    "brand": brand_name,
                    "image": img_name,
                    "fabric": best_match
    })
            except Exception as e:
                print(f"Skipping {img_name} due to {e}")

classifying runway fabrics using samples...


NameError: name 'os' is not defined

In [ ]:
# percentage
counts = Counter(results)
total = len(results)
print("\n--- FINAL FABRIC TRENDS ---")
for fabric, count in counts.items():
    print(f"{fabric}: {(count/total)*100:.1f}%")

JSON AND CSV EXTRACT

In [ ]:
df = pd.DataFrame(results)
df.to_csv(fabric_csv, index=False)

summary = dict(Counter(df['fabric']))
for_frontend = {
    "summary": summary,
    "percentages": {dist: round((v/len(df))*100,2) for dist, v in summary.items()},
    "details": results
}
json_output = "fabric_results.json"
with open(json_output, 'w') as f:
    json.dump(for_frontend, f, indent=4)

print("fabric analysis done and saved to csv and json paths")